In [ ]:
!nvidia-smi

Sat Jul 25 10:02:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             53W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
import torch, sys
print(f"Python: {sys.version.split()[0]} | Torch: {torch.__version__} | CUDA: {torch.version.cuda}")

Python: 3.12.13 | Torch: 2.11.0+cu128 | CUDA: 12.8


In [ ]:
!pip install -qU transformers peft trl datasets bitsandbytes accelerate huggingface_hub packaging ninja

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 130.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 885.0/885.0 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 771.9/771.9 kB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 48.7 MB/s eta 0:00:00


In [ ]:
!pip install -q --pre flash-attn-4

INFO: pip is looking at multiple versions of quack-kernels to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 382.7/382.7 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 MB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 112.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.7/338.7 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.8/897.8 kB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 30.0 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6

In [ ]:
MODEL = "0.5b"

MODEL_IDS = {
    "tiny": "tiiuae/falcon-h1-tiny-90m",
    "0.5b": "tiiuae/Falcon-H1-0.5B-Instruct",
}
MODEL_ID = MODEL_IDS[MODEL]
OUTPUT_DIR = f"falcon-h1-{MODEL}-pii-lora"
MERGED_DIR = f"falcon-h1-{MODEL}-pii-merged"
GGUF_F16 = f"falcon-h1-{MODEL}-pii_f16.gguf"
GGUF_Q5 = f"falcon-h1-{MODEL}-pii_q5_k_m.gguf"

print(f"Model: {MODEL_ID}")
print(f"LoRA output: {OUTPUT_DIR}")
print(f"GGUF final: {GGUF_Q5}")

Model: tiiuae/Falcon-H1-0.5B-Instruct
LoRA output: falcon-h1-0.5b-pii-lora
GGUF final: falcon-h1-0.5b-pii_q5_k_m.gguf


In [ ]:
import re
import torch
import random
from trl import SFTConfig
import os, sys, subprocess
from datasets import load_dataset, Dataset, concatenate_datasets
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    BitsAndBytesConfig, TrainingArguments
)
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer

In [ ]:
LLAMA_CPP_DIR = "/content/llama.cpp"

if not os.path.exists(LLAMA_CPP_DIR):
    !git clone --depth 1 https://github.com/ggml-org/llama.cpp.git
    !pip install -q llama.cpp/gguf-py/
    %cd llama.cpp
    !mkdir -p build && cd build && cmake .. -DLLAMA_CUDA=ON && cmake --build . --target quantize -j$(nproc)
    %cd /content
else:
    print("llama.cpp already cloned and built")

QUANTIZE_BIN = os.path.join(LLAMA_CPP_DIR, "build", "bin", "quantize")
CONVERT_SCRIPT = os.path.join(LLAMA_CPP_DIR, "convert_hf_to_gguf.py")
print(f"quantize binary: {os.path.exists(QUANTIZE_BIN)}")
print(f"convert script: {os.path.exists(CONVERT_SCRIPT)}")

llama.cpp already cloned and built
quantize binary: False
convert script: True


In [ ]:
%cd llama.cpp

/content/llama.cpp


In [ ]:
!pwd

/content/llama.cpp


In [ ]:
!cmake -B build -DGGML_CUDA=ON

CMAKE_BUILD_TYPE=Release
CMake Warning at CMakeLists.txt:153 (message):
  LLAMA_CUDA is deprecated, use GGML_CUDA instead
Call Stack (most recent call first):
  CMakeLists.txt:162 (llama_option_depr)


-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- x86 detected
-- Adding CPU backend variant ggml-cpu: -march=native 
-- CUDA Toolkit found
-- Using CMAKE_CUDA_ARCHITECTURES=80-real CMAKE_CUDA_ARCHITECTURES_NATIVE=80-real
-- CUDA host compiler is GNU 11.4.0
-- Including CUDA backend
-- ggml version: 0.17.0
-- ggml commit:  910196f
-- OpenSSL found: 3.0.2
-- Generating embedded license file for target: llama-app
-- Configuring done (0.2s)
-- Generating done (0.4s)
-- Build files have been written to: /content/llama.cpp/build


In [ ]:
!cmake --build build --target llama-quantize --config Release -j$(nproc)

[  0%] Building CXX object vendor/cpp-httplib/CMakeFiles/cpp-httplib.dir/httplib.cpp.o
[  0%] Building C object ggml/src/CMakeFiles/ggml-base.dir/ggml.c.o
[  0%] Building CXX object common/CMakeFiles/llama-common-base.dir/build-info.cpp.o
[  1%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml.cpp.o
[  1%] Building C object ggml/src/CMakeFiles/ggml-base.dir/ggml-alloc.c.o
[  1%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml-backend.cpp.o
[  1%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml-backend-meta.cpp.o
[  1%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml-opt.cpp.o
[  1%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml-threading.cpp.o
[  1%] Building C object ggml/src/CMakeFiles/ggml-base.dir/ggml-quants.c.o
[  3%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/gguf.cpp.o
[  3%] Linking CXX static library libllama-common-base.a
[  3%] Built target llama-common-base
[  3%] Linking CXX shared library ../../bin/libggm

PII Masking Dataset

In [ ]:
dataset = load_dataset("ai4privacy/pii-masking-200k", split="train")
print(f"Loaded {len(dataset)} examples")
print(f"Columns: {dataset.column_names}")
print()
print("=== Example ===")
print(f"source_text: {dataset[0]['source_text'][:200]}")
print(f"target_text: {dataset[0]['target_text'][:200]}")

Loaded 209261 examples
Columns: ['source_text', 'target_text', 'privacy_mask', 'span_labels', 'mbert_text_tokens', 'mbert_bio_labels', 'id', 'language', 'set']

=== Example ===
source_text: A student's assessment was found on device bearing IMEI: 06-184755-866851-3. The document falls under the various topics discussed in our Optimization curriculum. Can you please collect it?
target_text: A student's assessment was found on device bearing IMEI: [PHONEIMEI]. The document falls under the various topics discussed in our [JOBAREA] curriculum. Can you please collect it?


In [ ]:
PII_TAG_MAP = {
    "EMAIL": "email",
    "PHONENUMBER": "phone number",
    "FIRSTNAME": "first name",
    "LASTNAME": "last name",
    "PREFIX": "prefix",
    "MIDDLENAME": "middle name",
    "AGE": "age",
    "DOB": "date of birth",
    "DATE": "date",
    "TIME": "time",
    "GENDER": "gender",
    "SEX": "sex",
    "HEIGHT": "height",
    "EYECOLOR": "eye color",
    "PERSON": "person",
    "LOCATION": "location",
    "CITY": "city",
    "STATE": "state",
    "COUNTY": "county",
    "COUNTRY": "country",
    "STREET": "street",
    "BUILDINGNUMBER": "building number",
    "ZIPCODE": "zip code",
    "IPV4": "ip address",
    "IPV6": "ip address",
    "MAC": "mac address",
    "PASSWORD": "password",
    "USERAGENT": "user agent",
    "JOBTITLE": "job title",
    "JOBAREA": "job area",
    "JOBTYPE": "job type",
    "ORG": "organization",
    "COMPANYNAME": "company name",
    "ACCOUNTNAME": "account name",
    "ACCOUNTNUMBER": "account number",
    "CREDITCARDNUMBER": "credit card number",
    "CREDITCARDISSUER": "credit card issuer",
    "MASKEDNUMBER": "masked number",
    "PIN": "pin",
    "CURRENCY": "currency",
    "CURRENCYCODE": "currency code",
    "CURRENCYNAME": "currency name",
    "CURRENCYSYMBOL": "currency symbol",
    "AMOUNT": "amount",
    "IBAN": "iban",
    "PHONEIMEI": "phone imei",
    "VEHICLEVIN": "vehicle vin",
    "VEHICLEVRM": "vehicle vrm",
    "BITCOINADDRESS": "bitcoin address",
    "LITECOINADDRESS": "litecoin address",
    "ORDINALDIRECTION": "ordinal direction",
    "NEARBYGPSCOORDINATE": "gps coordinate",
}

def normalize_tags(text):
    for tag, replacement in PII_TAG_MAP.items():
        text = text.replace(f"[{tag}]", replacement)
    return text

IDENTITY_FRACTION = 0.20  # 20% of dataset size

n_identity = int(len(dataset) * IDENTITY_FRACTION)
target_texts = dataset["target_text"]

random.seed(42)
identity_samples = random.choices(target_texts, k=n_identity)

identity_dataset = Dataset.from_list([
    {"source_text": normalize_tags(t), "target_text": normalize_tags(t)}
    for t in identity_samples
])

augmented = concatenate_datasets([dataset, identity_dataset])
augmented = augmented.shuffle(seed=42)

splits = augmented.train_test_split(test_size=0.01, seed=42)
train_dataset = splits["train"]
eval_dataset = splits["test"]

print(f"Original: {len(dataset)} examples")
print(f"Augmented: {len(augmented)} examples (+{n_identity} identity)")
print(f"Train: {len(train_dataset)}  Eval: {len(eval_dataset)}")

Original: 209261 examples
Augmented: 251113 examples (+41852 identity)
Train: 248601  Eval: 2512


In [ ]:
def format_pii(example):
    messages = [
        {"role": "user", "content": example["source_text"]},
        {"role": "assistant", "content": example["target_text"]},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False)

LoRA setup beaches

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    llm_int8_skip_modules=["out_proj"],
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loading {MODEL_ID}...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    dtype=torch.bfloat16,
)
model.enable_input_require_grads()
model.config.use_cache = False

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["in_proj", "x_proj", "dt_proj"],
    modules_to_save=None,
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

Loading tiiuae/Falcon-H1-0.5B-Instruct...


[transformers] The fast path is not available because one of `(selective_state_update, causal_conv1d_fn, causal_conv1d_update)` is None. Falling back to the naive implementation. To install follow https://github.com/state-spaces/mamba/#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

trainable params: 2,520,576 || all params: 523,931,680 || trainable%: 0.4811


In [ ]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    # gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    max_steps=2000,
    learning_rate=2e-4,
    weight_decay=0.1,
    lr_scheduler_type="cosine",
    warmup_steps=50,
    packing=True,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    logging_steps=50,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    report_to="none",
    optim="adamw_8bit",
    eval_strategy="steps",
    eval_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    loss_type="nll",              # <-- standard NLL, not chunked
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    formatting_func=format_pii
)

Applying formatting function to train dataset:   0%|          | 0/248601 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/248601 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/248601 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/248601 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/248601 [00:00<?, ? examples/s]

Applying formatting function to eval dataset:   0%|          | 0/2512 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/2512 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/2512 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/2512 [00:00<?, ? examples/s]

Packing eval dataset:   0%|          | 0/2512 [00:00<?, ? examples/s]

You WANNA TRAIN?

In [ ]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 17, 'pad_token_id': 32768}.


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
50,2.539511,1.913502,1.899453,0.632269,404740.000000
100,1.669406,1.544001,1.516516,0.685160,809989.000000
150,1.481011,1.462080,1.481366,0.698597,1214122.000000
200,1.437130,1.419223,1.414283,0.705153,1618222.000000
250,1.392980,1.388202,1.373070,0.710930,2022744.000000
300,1.379509,1.365658,1.349197,0.714059,2426601.000000
350,1.344951,1.346463,1.331593,0.717113,2832370.000000
400,1.347952,1.332182,1.343158,0.719045,3236091.000000
450,1.324551,1.320835,1.338398,0.720870,3641047.000000
500,1.315023,1.309412,1.308836,0.722533,4046787.000000


KeyboardInterrupt: 

NOW its a sign of the times

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"LoRA saved to {OUTPUT_DIR}/")

LoRA saved to falcon-h1-0.5b-pii-lora/


In [ ]:
!pip install torchao --upgrade

In [ ]:
import torchao

In [ ]:
torchao.__version__

'0.17.0'

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

print("Loading LoRA adapter...")
merged_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
print("Merging...")
merged_model = merged_model.merge_and_unload()

merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print(f"Merged model saved to {MERGED_DIR}/")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

Loading LoRA adapter...
Merging...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model saved to falcon-h1-0.5b-pii-merged/


In [ ]:
random.seed(0)
demos = random.sample(range(len(eval_dataset)), 5)

print(f"{'INPUT':<70} {'EXPECTED':<70}")
print("-" * 140)
for idx in demos:
    ex = eval_dataset[idx]
    inp = ex['source_text'][:67]
    exp = ex['target_text'][:67]
    print(f"{inp:<70} {exp:<70}")

INPUT                                                                  EXPECTED                                                              
--------------------------------------------------------------------------------------------------------------------------------------------
Dear Mr.Berry LeslieHodkiewicz, Could you elaborate on the Human Ri    Dear [PREFIX][FIRSTNAME] [MIDDLENAME][LASTNAME], Could you elaborat   
Clovis, per una comunicazione senza intoppi durante il tuo programm    [FIRSTNAME], per una comunicazione senza intoppi durante il tuo pro   
Votre consultation génétique a été facturée à 0899464881954927 avec    Votre consultation génétique a été facturée à [CREDITCARDNUMBER] av   
Le rapport financier était bon, la prochaine fois assurez-vous qu'i    Le rapport financier était bon, la prochaine fois assurez-vous qu'i   
Gentile Stone.Renner, la tua candidatura per il nostro programma di    Gentile [USERNAME], la tua candidatura per il nostro programma di s   


In [ ]:
merged_model.eval()

def predict(text, max_new=128):
    messages = [{"role": "user", "content": text}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(merged_model.device)
    with torch.no_grad():
        outputs = merged_model.generate(
            **inputs,
            max_new_tokens=max_new,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response.strip()

print("Running inference on demo examples...\n")
for idx in demos:
    ex = eval_dataset[idx]
    inp = ex['source_text']
    exp = ex['target_text']
    pred = predict(inp)
    print("=" * 80)
    print(f"INPUT:    {inp}")
    print(f"EXPECTED: {exp}")
    print(f"PREDICTED: {pred}")
    print()

Running inference on demo examples...



[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


INPUT:    Dear Mr.Berry LeslieHodkiewicz, Could you elaborate on the Human Rights Law impact in Union County, Missouri? Please send your thoughts to Janis74@hotmail.com.
EXPECTED: Dear [PREFIX][FIRSTNAME] [MIDDLENAME][LASTNAME], Could you elaborate on the Human Rights Law impact in [COUNTY], [STATE]? Please send your thoughts to [EMAIL].
PREDICTED: Dear [PREFIX] [LASTNAME] [LASTNAME], Could you elaborate on the Human Rights Law impact in [COUNTY], [STATE]? Please send your thoughts to [EMAIL].

INPUT:    Clovis, per una comunicazione senza intoppi durante il tuo programma di scambio studentesco, fornisci il tuo 234.33.231.69 e Opera/9.7 (Macintosh; Intel Mac OS X 10.8.7 U; JA Presto/2.9.169 Version/11.00).
EXPECTED: [FIRSTNAME], per una comunicazione senza intoppi durante il tuo programma di scambio studentesco, fornisci il tuo [IP] e [USERAGENT].
PREDICTED: [FIRSTNAME], per una comunicazione senza intoppi durante il tuo programma di scambio studentesco, fornisci il tuo [IPV4] e [USERA

Convert to GGUF and let's save it all

In [ ]:
!python {CONVERT_SCRIPT} {MERGED_DIR} --outfile {GGUF_F16} --outtype f16
f16_size = os.path.getsize(GGUF_F16) / 1e9
print(f"GGUF FP16: {GGUF_F16} ({f16_size:.2f} GB)")

INFO:hf-to-gguf:Loading model: falcon-h1-0.5b-pii-merged
INFO:numexpr.utils:NumExpr defaulting to 12 threads.
INFO:hf-to-gguf:Model architecture: FalconH1ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,             torch.bfloat16 --> F16, shape = {1024, 32784}
INFO:hf-to-gguf:token_embd.weight,         torch.bfloat16 --> F16, shape = {1024, 32784}
INFO:hf-to-gguf:output_norm.weight,        torch.bfloat16 --> F32, shape = {1024}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.bfloat16 --> F16, shape = {2048, 1024}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.bfloat16 --> F16, shape = {1024, 2048}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.bfloat16 --> F16, shape = {1024, 2048}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.bfloat16 --> F32, shape = {1024}
INFO:hf-to-gguf:blk.0.ssm_a,               torch.bfloat16 --> 

In [ ]:
QUANTIZE_BIN

'/content/llama.cpp/build/bin/quantize'

In [ ]:
!ls -al

total 1020316
drwxr-xr-x  1 root root       4096 Jul 25 13:13  .
drwxr-xr-x  1 root root       4096 Jul 25 09:47  ..
-rw-r--r--  1 root root         91 Jul 25 13:09 '=0.16.0'
drwxr-xr-x  4 root root       4096 Jun  4 13:39  .config
-rw-r--r--  1 root root 1044764416 Jul 25 13:13  falcon-h1-0.5b-pii_f16.gguf
drwxr-xr-x  4 root root       4096 Jul 25 13:08  falcon-h1-0.5b-pii-lora
drwxr-xr-x  2 root root       4096 Jul 25 13:11  falcon-h1-0.5b-pii-merged
drwxr-xr-x 31 root root       4096 Jul 25 10:05  llama.cpp
drwxr-xr-x  1 root root       4096 Jun  4 13:39  sample_data


In [ ]:
!ls /content/llama.cpp/build/bin/

In [ ]:
!/content/llama.cpp/build/bin/llama-quantize /content/falcon-h1-0.5b-pii_f16.gguf /content/falcon-h1-0.5b-pii_q5_k_m.gguf "Q5_K_M"

ggml_cuda_init: found 1 CUDA devices (Total VRAM: 40441 MiB):
  Device 0: NVIDIA A100-SXM4-40GB, compute capability 8.0, VMM: yes, VRAM: 40441 MiB
llama_print_build_info: build = 1 (910196f)
llama_print_build_info: built with GNU 11.4.0 for Linux x86_64
llama_quantize: quantizing '/content/falcon-h1-0.5b-pii_f16.gguf' to '/content/falcon-h1-0.5b-pii_q5_k_m.gguf' as Q5_K_M
llama_model_loader: loaded meta data with 33 key-value pairs and 579 tensors from /content/falcon-h1-0.5b-pii_f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = falcon-h1
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Falcon H1 0.5b Pii Merged
llama_model_loader: - kv   3:                           general

In [ ]:
!{QUANTIZE_BIN} {GGUF_F16} {GGUF_Q5} Q5_K_M
q5_size = os.path.getsize(GGUF_Q5) / 1e9
print(f"GGUF Q5_K_M: {GGUF_Q5} ({q5_size:.2f} GB)")

/bin/bash: line 1: /content/llama.cpp/build/bin/quantize: No such file or directory


FileNotFoundError: [Errno 2] No such file or directory: 'falcon-h1-0.5b-pii_q5_k_m.gguf'

In [ ]:
from google.colab import files
print("Downloading Q5_K_M GGUF...")
files.download(GGUF_Q5)